In [1]:
%load_ext autoreload
%autoreload 2
from omegaconf import OmegaConf
from oat.data_module import factory

# Brief description about dataset and loader

## Dataset
The dataset collects fetches all the different data sources and does some precomputation, such that they can be loaded efficiently later.

In particular, it caches stationary variables, such that they are already in memory.

Dynamic variables were already rechunked in time, so time-slices can be read in with minimal effort.

In the end the dataset provides a dictionary with 
    - global information (NWP + TOPO at low resolution)
    - observations (measurements at a station + some high res topography info)
    - query points (some high res topography info + (optional) measurements required for validation/testing)

## Dataloader

The dataloader similar to the default pytorch loader, but additional applies some data transforms (e.g. log or scaling) on the fly. Transforms can be defined in the config.yaml file.

For different settings see `config/data/wind_data.yaml`.

## Things that need to be done still:

- How should the stations be split between observations/queries and train/val/test
- Can validation condition on training stations? And test on training and validation?
- Dynamically set query points?
- At the moment cached is saved where the code is executed. You want to specify that.
- Anything else?


In [6]:
from torch.utils.data import DataLoader
data_config = OmegaConf.load('/scratch/mch/cdonner/observation_assimilation_transformer/configs/data/wind_data.yaml')
dataset = factory.create_dataset(data_config)
#train_dataloader, val_dataloader, test_dataloader = factory.create_dataset(data_config)
#batch = next(iter(train_dataloader))

loader = DataLoader(dataset, batch_size=16, shuffle=False)


In [7]:
batch = next(iter(loader))
print(batch['index'])

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])


In [10]:
dataset.get_query(
    idx=0,
    query_coords=batch['observation_data']['coords'][0],
)

{'coords': tensor([[2609371.5000, 1148939.1250],
         [2688729.0000, 1220955.6250],
         [2560403.2500, 1130713.7500],
         [2690180.7500, 1193563.0000],
         [2752692.2500, 1164037.2500],
         [2687444.7500, 1165043.1250],
         [2760386.7500, 1261382.2500],
         [2771031.0000, 1184830.0000],
         [2586853.5000, 1105302.3750],
         [2610909.7500, 1265611.7500],
         [2798422.5000, 1143019.5000],
         [2601934.5000, 1204409.5000],
         [2659813.5000, 1267695.8750],
         [2718535.0000, 1132791.5000],
         [2515887.0000, 1153212.2500],
         [2657996.0000, 1135420.8750],
         [2771282.5000, 1148118.8750],
         [2737713.2500, 1263612.8750],
         [2629563.7500, 1141083.5000],
         [2595831.0000, 1163586.8750],
         [2555262.7500, 1138175.7500],
         [2536981.7500, 1203975.8750],
         [2647552.7500, 1176802.2500],
         [2816493.7500, 1170255.0000],
         [2648396.7500, 1248369.5000],
         [25509

In [8]:
print(batch['observation_data'])

{'coords': tensor([[[2609371.5000, 1148939.1250],
         [2688729.0000, 1220955.6250],
         [2560403.2500, 1130713.7500],
         ...,
         [2780616.0000, 1189634.0000],
         [2626406.0000, 1233849.0000],
         [2624297.2500, 1097573.8750]],

        [[2609371.5000, 1148939.1250],
         [2688729.0000, 1220955.6250],
         [2560403.2500, 1130713.7500],
         ...,
         [2780616.0000, 1189634.0000],
         [2626406.0000, 1233849.0000],
         [2624297.2500, 1097573.8750]],

        [[2609371.5000, 1148939.1250],
         [2688729.0000, 1220955.6250],
         [2560403.2500, 1130713.7500],
         ...,
         [2780616.0000, 1189634.0000],
         [2626406.0000, 1233849.0000],
         [2624297.2500, 1097573.8750]],

        ...,

        [[2609371.5000, 1148939.1250],
         [2688729.0000, 1220955.6250],
         [2560403.2500, 1130713.7500],
         ...,
         [2780616.0000, 1189634.0000],
         [2626406.0000, 1233849.0000],
         [262429

# Batch Structure

| Data Group | Field | Shape | Description | Paper Notation |
|------------|-------|-------|-------------|----------------|
| **global_data** | global_field | [B, C, H_lr, W_lr] | Coarse NWP fields + low-res topography | U_nwp |
| | time_emb | [B, C_time] | Time features: sin/cos(diurnal, annual) | τ |
| **observation_data (KV)** | coords | [B, N_kv, 2] | Station coordinates on Swiss grid | x_kv |
| | geo | [B, N_kv, C_geo] | Local geometry: position, elevation | g_kv |
| | topo | [B, N_kv, C_topo, H_hr, W_hr] | High-res topographic patches | ξ_kv |
| | obs | [B, N_kv, C_y] | Station measurements | y_kv |
| | mask_obs | [B, N_kv, C_y] | Binary mask for valid observations | M_obs |
| **query_data** | coords | [B, N_q, 2] | Query point coordinates | x_q |
| | geo | [B, N_q, C_geo] | Local geometry features | g_q |
| | topo | [B, N_q, C_topo, H_hr, W_hr] | High-res topographic patches | ξ_q |
| | obs | [B, N_q, C_y] | (Optional) Measurements at query points | - |
| | mask_obs | [B, N_q, C_y] | (Optional) Observation mask | - |

---

### Notation Mapping
- **B**: Batch size
- **N_kv**: Number of observation (station) points
- **N_q**: Number of query points
- **C, C_y, C_geo, C_topo, C_time**: Channel dimensions
- **H_lr, W_lr**: Low-resolution spatial dimensions
- **H_hr, W_hr**: High-resolution patch dimensions

## Global Data

In [ ]:
from matplotlib import pyplot as plt
idx = 0
global_vars = data_config.vars.nwp_vars + data_config.vars.topo_vars
for chan_idx, global_var in enumerate(global_vars):
    plt.pcolor(batch['global_data']['global_field'][idx,chan_idx].cpu(), cmap='bwr')
    plt.colorbar(label=global_var)
    plt.show()

In [ ]:
plt.scatter(batch['query_data']['coords'][idx,:,0].cpu(), batch['query_data']['coords'][idx,:,1].cpu(), c='blue', label='Queries')
plt.scatter(batch['observation_data']['coords'][idx,:,0].cpu(), batch['observation_data']['coords'][idx,:,1].cpu(), c='red', label='Observations')
plt.xlabel('Swiss Grid Easting')
plt.ylabel('Swiss Grid Northing')
plt.title('Query Data Coordinates')
plt.legend()
plt.show()

In [ ]:
for chan_idx, variable in enumerate(data_config.vars.obs_vars):
    plt.scatter(batch['observation_data']['coords'][idx,:,0].cpu(), batch['observation_data']['coords'][idx,:,1].cpu(), c=batch['observation_data']['obs'][idx,:,chan_idx].cpu(), cmap='inferno')
    plt.colorbar(label=variable)
    plt.xlabel('Swiss Grid Easting')
    plt.ylabel('Swiss Grid Northing')
    plt.title('Observation Data Coordinates with Values')
plt.show()

## Topo data around station

### At observed station

In [ ]:
# For observation
station_idx = 10
dem_idx = np.where(np.array(data_config.vars.topo_vars) == 'DEM')[0][0]
plt.pcolor(batch['observation_data']['topo'][idx, station_idx, dem_idx].cpu(), cmap='terrain')
plt.colorbar(label='Elevation (m)')
plt.title('Topography')

### At query point

In [ ]:
station_idx = 10
dem_idx = np.where(np.array(data_config.vars.topo_vars) == 'DEM')[0][0]
plt.pcolor(batch['query_data']['topo'][idx, station_idx, dem_idx].cpu(), cmap='terrain')
plt.colorbar(label='Elevation (m)')
plt.title('Topography')